# Coffee Shop ERP — Data Exploration
Quick visual pass over items, costing, and inventory using pandas + matplotlib.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (9, 5)

conn = sqlite3.connect('../data/coffee_shop.db')

## 1. Items overview

In [ ]:
items = pd.read_sql_query('SELECT * FROM items', conn)
items.groupby('item_type').size().rename('count')

In [ ]:
items[items.item_type == 'finished_good'][['item_code', 'item_name', 'standard_cost', 'sale_price']]

## 2. Margin by finished good

In [ ]:
margin = pd.read_sql_query('SELECT * FROM v_item_margin ORDER BY margin_pct DESC', conn)
margin

In [ ]:
fig, ax = plt.subplots()
ax.bar(margin['item_name'], margin['margin_pct'] * 100, color='#6f4e37')
ax.set_ylabel('Gross margin %')
ax.set_title('Gross Margin % by Finished Good')
ax.tick_params(axis='x', rotation=45)
for label in ax.get_xticklabels():
    label.set_ha('right')
plt.tight_layout()
plt.show()

## 3. Cost breakdown: material vs. labor
Uses the same rollup logic as `costing.py`, computed here directly in pandas for exploration.

In [ ]:
formulas = pd.read_sql_query('SELECT * FROM formulas', conn)
routes = pd.read_sql_query('SELECT * FROM routes', conn)
route_ops = pd.read_sql_query('SELECT * FROM route_operations', conn)
workstations = pd.read_sql_query('SELECT * FROM workstations', conn)

# labor cost per item: join routes -> route_operations -> workstations
labor = (route_ops.merge(routes, on='route_id')
                   .merge(workstations, on='workstation_id', how='left'))
labor['op_cost'] = (labor['setup_minutes'] + labor['run_minutes']) / 60 * labor['hourly_rate'].fillna(0)
labor_by_item = labor.groupby('item_id')['op_cost'].sum().rename('labor_cost')

# material cost per item: one level of formulas, using items.standard_cost as component cost
# (already rolled up bottom-up by costing.py, so this reflects the full multi-tier BOM)
comp_cost = items.set_index('item_id')['standard_cost']
formulas['component_cost'] = formulas['component_item_id'].map(comp_cost)
formulas['line_cost'] = formulas['quantity'] * (1 + formulas['scrap_pct']) * formulas['component_cost']
material_by_item = formulas.groupby('parent_item_id')['line_cost'].sum().rename('material_cost')

cost_breakdown = (items.set_index('item_id')[['item_code', 'item_type']]
                        .join(material_by_item)
                        .join(labor_by_item)
                        .fillna(0))
cost_breakdown = cost_breakdown[cost_breakdown.item_type.isin(['intermediate', 'finished_good'])]
cost_breakdown = cost_breakdown.sort_values(['item_type', 'material_cost'], ascending=[True, False])
cost_breakdown

In [ ]:
fig, ax = plt.subplots()
fg_breakdown = cost_breakdown[cost_breakdown.item_type == 'finished_good']
cost_breakdown_sorted = fg_breakdown.sort_values('material_cost', ascending=False)
ax.bar(cost_breakdown_sorted['item_code'], cost_breakdown_sorted['material_cost'], label='Material', color='#a67b5b')
ax.bar(cost_breakdown_sorted['item_code'], cost_breakdown_sorted['labor_cost'],
       bottom=cost_breakdown_sorted['material_cost'], label='Labor', color='#3e2723')
ax.set_ylabel('Cost ($)')
ax.set_title('Standard Cost Breakdown: Material vs. Labor (Finished Goods)')
ax.tick_params(axis='x', rotation=45)
for label in ax.get_xticklabels():
    label.set_ha('right')
ax.legend()
plt.tight_layout()
plt.show()

### 3b. Cost breakdown at the SFG (intermediate) level
This is the cost baked into each base before add-ons/finishing turn it into a sellable finished good.

In [ ]:
sfg_breakdown = cost_breakdown[cost_breakdown.item_type == 'intermediate'].sort_values('material_cost', ascending=False)

fig, ax = plt.subplots()
ax.bar(sfg_breakdown['item_code'], sfg_breakdown['material_cost'], label='Material', color='#a67b5b')
ax.bar(sfg_breakdown['item_code'], sfg_breakdown['labor_cost'],
       bottom=sfg_breakdown['material_cost'], label='Labor', color='#3e2723')
ax.set_ylabel('Cost ($)')
ax.set_title('Standard Cost Breakdown: Material vs. Labor (SFG / Intermediate)')
ax.tick_params(axis='x', rotation=45)
for label in ax.get_xticklabels():
    label.set_ha('right')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Inventory: on-hand vs. reorder point

In [ ]:
inv = pd.read_sql_query('''
    SELECT it.item_code, i.on_hand_qty, i.reorder_point, i.reorder_qty, i.pack_size, i.min_order_qty
    FROM inventory i JOIN items it ON it.item_id = i.item_id
    ORDER BY it.item_code
''', conn)
inv

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
y = range(len(inv))
ax.barh(y, inv['on_hand_qty'], color='#c8a97e', label='On hand')
ax.scatter(inv['reorder_point'], y, color='crimson', zorder=3, label='Reorder point')
ax.set_yticks(list(y))
ax.set_yticklabels(inv['item_code'])
ax.set_xlabel('Quantity (item UOM)')
ax.set_title('On-Hand Inventory vs. Reorder Point')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Inventory value by item

In [ ]:
inv_value = pd.read_sql_query('SELECT * FROM v_inventory_value ORDER BY inventory_value DESC', conn)
inv_value

In [ ]:
fig, ax = plt.subplots()
ax.bar(inv_value['item_name'], inv_value['inventory_value'], color='#556b2f')
ax.set_ylabel('Inventory value ($)')
ax.set_title(f"Inventory Value by Raw Material (Total: ${inv_value['inventory_value'].sum():,.2f})")
ax.tick_params(axis='x', rotation=60)
for label in ax.get_xticklabels():
    label.set_ha('right')
plt.tight_layout()
plt.show()

## 7. Fulfillment & Substitutions
What happened when a customer's first choice was out of stock: did they take a substitute, or leave without ordering?

In [ ]:
fulfillment_summary = pd.read_sql_query('''
    SELECT
        CASE
            WHEN original_item_id IS NOT NULL THEN 'substituted'
            WHEN fulfilled = 1 THEN 'fulfilled_direct'
            ELSE 'stockout_no_sale'
        END AS outcome,
        COUNT(*) AS n
    FROM order_lines
    GROUP BY outcome
''', conn)
fulfillment_summary

In [ ]:
fig, ax = plt.subplots()
colors = {'fulfilled_direct': '#556b2f', 'substituted': '#c8a97e', 'stockout_no_sale': '#b22222'}
ax.bar(fulfillment_summary['outcome'], fulfillment_summary['n'],
       color=[colors[o] for o in fulfillment_summary['outcome']])
ax.set_ylabel('Order lines')
ax.set_title('Order Line Outcomes')
for i, n in enumerate(fulfillment_summary['n']):
    ax.text(i, n, f'{n:,}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

### 7a. Substitution pairs
What customers originally wanted vs. what they actually walked away with.

In [ ]:
substitutions = pd.read_sql_query('''
    SELECT orig.item_code AS wanted, new.item_code AS got, COUNT(*) AS n
    FROM order_lines ol
    JOIN items orig ON orig.item_id = ol.original_item_id
    JOIN items new ON new.item_id = ol.item_id
    WHERE ol.original_item_id IS NOT NULL
    GROUP BY wanted, got
    ORDER BY n DESC
''', conn)
substitutions

### 7b. Which items cause the most substitutions away from them?
A proxy for which items run out most relative to demand -- worth cross-referencing with the inventory reorder analysis in section 4.

In [ ]:
lost_demand_by_item = (substitutions.groupby('wanted')['n'].sum()
                                     .sort_values(ascending=False))

fig, ax = plt.subplots()
ax.bar(lost_demand_by_item.index, lost_demand_by_item.values, color='#b22222')
ax.set_ylabel('Times customers were substituted away from this item')
ax.set_title('Stockout-Driven Substitution, by Originally Requested Item')
ax.tick_params(axis='x', rotation=45)
for label in ax.get_xticklabels():
    label.set_ha('right')
plt.tight_layout()
plt.show()

### 7c. Fully lost sales (no substitute taken)

In [ ]:
lost_sales = pd.read_sql_query('''
    SELECT it.item_code, o.order_date, ol.quantity, ol.unit_price
    FROM order_lines ol
    JOIN items it ON it.item_id = ol.item_id
    JOIN orders o ON o.order_id = ol.order_id
    WHERE ol.fulfilled = 0
    ORDER BY o.order_date
''', conn)
print(f"Total lost revenue: ${(lost_sales['quantity'] * lost_sales['unit_price']).sum():,.2f}")
lost_sales

## 8. Orders overview
Now populated by simulate_orders.py + inventory_engine.py.


In [ ]:
orders = pd.read_sql_query('SELECT * FROM orders', conn)
order_lines = pd.read_sql_query('SELECT * FROM order_lines', conn)
print(f'{len(orders)} orders, {len(order_lines)} order lines')
orders.head()

In [ ]:
conn.close()

## 9. Phase 6: Working Capital Optimization
Trading off service level (fulfillment rate) against inventory dollars tied up, found by re-simulating the actual system at different safety-stock levels (see `src/optimize_inventory.py`).

In [ ]:
trials = pd.read_csv('../data/results/optimization_trials.csv').sort_values('z')
trials

In [ ]:
fig, ax = plt.subplots()
ax.plot(trials['avg_inventory_value'], trials['fulfillment_rate'] * 100, 'o-', color='#6f4e37')
ax.axhline(97, color='crimson', linestyle='--', label='97% target')
ax.set_xlabel('Average Inventory Value ($)')
ax.set_ylabel('Fulfillment Rate (%)')
ax.set_title('Efficient Frontier: Working Capital vs. Service Level')
ax.legend()
plt.tight_layout()
plt.show()

Each point is a full re-simulation of 60 days of real order history under a different safety-stock policy (z-score). The curve shows sharply diminishing returns -- the last few points of fulfillment (99% -> 100%) cost disproportionately more working capital than the jump from 96% -> 98%.